# БОРАН — четыре алгоритма шаг за шагом

Финальный проект по AI Fundamentals. Ноутбук показывает ту же цепочку, что и сайт: **HMM → Байес → A\* → MDP**. Каждый шаг: вход → расчёт → выход.

Все вероятности, веса и награды — допущения команды, а не обученная модель.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from boran import hmm, bayes, astar, mdp, weather, graph, config

## 1. HMM: какое покрытие сейчас

**Скрытые состояния:** сухо, мокро, свежий снег, накат, гололёд. **Наблюдения** получаем из погоды по часам. Сценарий: днём оттепель, ночью мороз.

In [2]:
wx = weather.scenario('thaw_freeze')
for h in wx['past']:
    print(h['time'], f"{h['temp']:+5.1f}°", 'осадки' if h['snow'] >= 0.1 else '')

20:00  +3.0° 
21:00  +4.0° 
22:00  +4.0° осадки
23:00  +3.0° осадки
00:00  +2.0° 
01:00  +1.0° 
02:00  +0.0° 
03:00  -2.0° 
04:00  -4.0° 
05:00  -6.0° 
06:00  -8.0° 
07:00  -9.0° 


In [3]:
obs = hmm.observe(wx['past'])
print('наблюдения:', [hmm.OBS[o] for o in obs])

наблюдения: ['thaw_dry', 'thaw_dry', 'thaw_wet', 'thaw_wet', 'thaw_dry', 'zero_cross', 'zero_cross', 'frost_dry', 'frost_dry', 'frost_dry', 'frost_dry', 'frost_dry']


**Forward-фильтр** считает P(состояние | все наблюдения до этого часа):

$$b_t(j) \propto e_j(o_t)\sum_i b_{t-1}(i)\,a_{ij}$$

In [4]:
beliefs = hmm.forward(obs)
print('час   ' + '  '.join(f'{s:>6}' for s in hmm.STATES))
for h, b in zip(wx['past'], beliefs):
    print(h['time'], '  '.join(f'{p:6.2f}' for p in b))

час      dry     wet    snow  packed     ice
20:00   0.84    0.07    0.02    0.05    0.02
21:00   0.91    0.07    0.01    0.01    0.01
22:00   0.29    0.60    0.09    0.01    0.01
23:00   0.06    0.87    0.04    0.01    0.02
00:00   0.42    0.52    0.01    0.01    0.04
01:00   0.22    0.29    0.04    0.02    0.44
02:00   0.09    0.14    0.04    0.03    0.71
03:00   0.16    0.00    0.03    0.09    0.72
04:00   0.19    0.00    0.03    0.13    0.65
05:00   0.22    0.00    0.03    0.16    0.58
06:00   0.24    0.00    0.03    0.19    0.53
07:00   0.27    0.00    0.03    0.21    0.49


**Витерби** находит самую вероятную историю состояний целиком.

In [5]:
path, logp = hmm.viterbi(obs)
print('путь Витерби:', ' → '.join(hmm.STATES[i] for i in path))
print('log P =', round(logp, 3))
now = beliefs[-1]
print('сейчас:', {s: round(p, 3) for s, p in zip(hmm.STATES, now)})

путь Витерби: dry → dry → wet → wet → wet → ice → ice → ice → ice → ice → ice → ice
log P = -18.863
сейчас: {'dry': 0.266, 'wet': 0.002, 'snow': 0.035, 'packed': 0.206, 'ice': 0.492}


## 2. Наивный Байес: опасен ли участок

Prior берём из HMM: P(скользко) = P(гололёд) + 0.4·P(накат) + 0.15·P(снег) + 0.1·P(мокро). Затем учитываем признаки улицы.

In [6]:
prior = bayes.prior_from_hmm(now)
print('prior P(скользко) =', round(prior, 4))
print('P(признак | скользко), P(признак | не скользко):')
for f, (a, b) in bayes.LIKELIHOOD.items():
    print(f'  {f:10} {a:.2f}  {b:.2f}')

prior P(скользко) = 0.5799
P(признак | скользко), P(признак | не скользко):
  shade      0.55  0.30
  open       0.50  0.35
  uncleaned  0.70  0.40
  stop       0.25  0.10
  bridge     0.10  0.03


In [7]:
street = {'shade': True, 'open': False, 'uncleaned': True, 'stop': False, 'bridge': False}
res = bayes.explain(prior, street)
for s in res['steps']:
    print(f"{s['feature']:10} {'есть' if s['present'] else 'нет ':4}  → P = {s['after']:.4f}")
print('posterior P(скользко | признаки) =', res['posterior'])

shade      есть  → P = 0.7168
open       нет   → P = 0.6607
uncleaned  есть  → P = 0.7731
stop       нет   → P = 0.7395
bridge     нет   → P = 0.7248
posterior P(скользко | признаки) = 0.7248


In [8]:
main_road = {'shade': False, 'open': True, 'uncleaned': False, 'stop': False, 'bridge': False}
print('двор в тени:      ', round(bayes.posterior(prior, street), 3))
print('чищеный проспект: ', round(bayes.posterior(prior, main_road), 3))

двор в тени:       0.725
чищеный проспект:  0.329


## 3. A\*: безопасный маршрут

Маленький граф из 5 перекрёстков. Прямая дорога 0→1→2 короче (280 м), но скользкая. Обход 0→3→4→2 длиннее (440 м), но чистый.

Стоимость участка = длина × (1 + α·P(скользко) + β·ветер), α = 4. Эвристика h — расстояние по прямой, она никогда не завышает стоимость.

In [9]:
nodes = [[71.400, 51.100], [71.402, 51.100], [71.404, 51.100], [71.401, 51.1012], [71.403, 51.1012]]
def edge(u, v, length):
    return {'u': u, 'v': v, 'len': length, 'mask': 0, 'warm': False, 'hw': 'footway', 'p': [nodes[u], nodes[v]]}
edges = [edge(0, 1, 140), edge(1, 2, 140), edge(0, 3, 150), edge(3, 4, 140), edge(4, 2, 150)]
city = graph.City({'bbox': [51.09, 71.39, 51.11, 71.41], 'nodes': nodes, 'edges': edges, 'buildings': [], 'water': {'lines': [], 'polys': []}, 'stops': [], 'warm': []})
risks = [0.9, 0.9, 0.05, 0.05, 0.05]
for e, r in zip(edges, risks):
    print(f"{e['u']}–{e['v']}: {e['len']} м, P = {r}, стоимость = {astar.edge_cost(e, r, 0.0):.0f}")

0–1: 140 м, P = 0.9, стоимость = 644
1–2: 140 м, P = 0.9, стоимость = 644
0–3: 150 м, P = 0.05, стоимость = 180
3–4: 140 м, P = 0.05, стоимость = 168
4–2: 150 м, P = 0.05, стоимость = 180


In [10]:
short = astar.search(city, 0, 2, risks, 0.0, safe=False)
safe = astar.search(city, 0, 2, risks, 0.0, safe=True)
print('короткий:   ', short['nodes'], 'длина', short['cost'], 'раскрыто узлов', len(short['order']))
print('безопасный: ', safe['nodes'], 'стоимость', round(safe['cost'], 1), 'раскрыто узлов', len(safe['order']))
for name, r in (('короткий', short), ('безопасный', safe)):
    d = astar.describe(city, r, risks, 0.0)
    print(f"{name}: {d['length']} м, {d['minutes']} мин, риск упасть {d['fall'] * 100:.2f}%")

короткий:    [0, 1, 2] длина 280.0 раскрыто узлов 3
безопасный:  [0, 3, 4, 2] стоимость 528.0 раскрыто узлов 4
короткий: 280 м, 3.8 мин, риск упасть 1.25%
безопасный: 440 м, 5.3 мин, риск упасть 0.11%


### 3b. Пять алгоритмов поиска на настоящих улицах Астаны

Граф пешеходных улиц всей Астаны из OpenStreetMap. Маршрут: кампус Qairu → Хан Шатыр. Одинаковые точки, одинаковый граф, разные стратегии.

- **BFS** — очередь, раскрывает слоями, минимум участков.
- **DFS** — стек, уходит вглубь, путь не оптимален.
- **UCS** (Дейкстра) — очередь с приоритетом по стоимости g.
- **Greedy** — приоритет по эвристике h (расстояние по прямой).
- **A\*** — приоритет f = g + h.

In [11]:
from boran import pipeline, search
city = graph.get_city()
print('перекрёстков:', len(city.nodes), '| участков улиц:', len(city.edges))
an = pipeline.analyse(city, weather.scenario('thaw_freeze'))
start = city.nearest_node(71.417377, 51.089187)
goal = city.nearest_node(71.4036, 51.1323)
res = search.compare(city, an, start, goal)
print(f"{'алгоритм':8} {'раскрыто':>9} {'участков':>9} {'длина, км':>10} {'риск, %':>8} {'стоимость':>10} {'мс':>6}")
for r in res:
    print(f"{r['algo']:8} {r['expanded']:9d} {r['steps']:9d} {r['length'] / 1000:10.1f} {r['fall'] * 100:8.1f} {r['cost']:10.0f} {r['ms']:6.1f}", '← оптимально' if r['best_cost'] else '')

перекрёстков: 59070 | участков улиц: 85491


алгоритм  раскрыто  участков  длина, км  риск, %  стоимость     мс
bfs          30283       104        6.8     14.4      19900   33.8 
dfs          56371      2312      123.4     95.5     357582   71.3 
ucs          20679       148        6.9      9.6      13472   63.1 ← оптимально
greedy         154       149        6.5     14.8      19161    0.8 
astar        15688       148        6.9      9.6      13472   68.5 ← оптимально


UCS и A\* находят один и тот же самый дешёвый путь, но A\* раскрывает меньше перекрёстков благодаря эвристике. BFS находит путь с наименьшим числом участков, но он длиннее и опаснее. Greedy думает быстрее всех, но его путь дороже. DFS находит путь, но это огромный крюк по всему району.

## 4. MDP: что делать сейчас

**Состояния:** спокойно / снег / буран. **Действия:** идти, ждать 20 минут, транспорт. Награда — минус «минуты потерь»: время + холод + риск упасть.

$$V(s)=\max_a\Big[R(s,a)+\gamma\sum_{s'}P(s'\mid s,a)\,V(s')\Big]$$

Сценарий: буран 18 м/с, по прогнозу стихает. Идти пешком 30 минут.

In [12]:
bl = weather.scenario('blizzard')
sev_now, sev_next = mdp.severity(bl['now']), mdp.severity(bl['ahead'][2])
level, trend = mdp.level(sev_now), mdp.trend_from_forecast(sev_now, sev_next)
print('тяжесть сейчас', round(sev_now, 2), '→ через 3 часа', round(sev_next, 2))
print('уровень:', level, '| тренд:', trend)
print('P(следующий уровень):', dict(zip(mdp.LEVELS, mdp.TREND_P[trend][level])))

тяжесть сейчас 0.92 → через 3 часа 0.49
уровень: blizzard | тренд: ease
P(следующий уровень): {'calm': 0.15, 'snow': 0.5, 'blizzard': 0.35}


In [13]:
inp = {'walk_min': 30.0, 'walk_risk': 0.027, 'ride_min': 6.0, 'access_min': 6.0, 'access_risk': 0.007, 'cold_factor': mdp.cold_factor(bl['now']['apparent'])}
res = mdp.solve(inp, level, trend)
print('cold_factor =', round(inp['cold_factor'], 3))
print('уровень     ' + '  '.join(f'{a:>8}' for a in mdp.ACTIONS))
for s in mdp.LEVELS:
    q = res['table'][s]
    print(f'{s:10}  ' + '  '.join(f"{q.get(a, float('nan')):8.1f}" for a in mdp.ACTIONS), ' ← сейчас' if s == level else '')
print('лучшее действие:', res['best'])

cold_factor = 0.533
уровень           go      wait   transit
calm           -47.6     -53.5     -34.3 
snow           -59.4     -58.9     -44.0 
blizzard       -91.8     -69.0     -79.4  ← сейчас
лучшее действие: wait


In [14]:
calm = mdp.solve(dict(inp, cold_factor=0.1, walk_min=8.0, walk_risk=0.01), 'calm', 'steady')
print('короткий путь в спокойную погоду:', calm['q'], '→', calm['best'])

короткий путь в спокойную погоду: {'go': -9.4, 'transit': -28.7, 'wait': -29.2} → go


## 5. Народная карта гололёда: Байес + HMM по отметкам людей

Каждая отметка — наблюдение для скрытого состояния участка (скользко / нет). Prior — прогноз Байеса по погоде. Между отметками вероятность возвращается к prior (τ = 3 ч).

| отметка | P(отметка ∣ скользко) | P(отметка ∣ не скользко) |
|---|---|---|
| скользко | 0.70 | 0.08 |
| снег не убран | 0.25 | 0.22 |
| чисто / посыпано | 0.05 | 0.70 |

In [15]:
from boran import reports
import time
now = time.time()
prior = 0.25
marks = [(now - 3 * 3600, 'ice'), (now - 2.5 * 3600, 'ice'), (now - 40 * 60, 'dry')]
for h in (3.5, 3.0, 2.5, 2.0, 1.0, 0.7, 0.5, 0.0):
    t = now - h * 3600
    seen = [m for m in marks if m[0] <= t]
    print(f'за {h:3.1f} ч до сейчас  отметок {len(seen)}  P(скользко) = {reports.belief(prior, seen, t):.2f}')

за 3.5 ч до сейчас  отметок 0  P(скользко) = 0.25
за 3.0 ч до сейчас  отметок 1  P(скользко) = 0.74
за 2.5 ч до сейчас  отметок 2  P(скользко) = 0.95
за 2.0 ч до сейчас  отметок 2  P(скользко) = 0.84
за 1.0 ч до сейчас  отметок 2  P(скользко) = 0.67
за 0.7 ч до сейчас  отметок 2  P(скользко) = 0.63
за 0.5 ч до сейчас  отметок 3  P(скользко) = 0.12
за 0.0 ч до сейчас  отметок 3  P(скользко) = 0.14


Где больше всего ходят люди — оцениваем выборкой кратчайших путей между остановками и местами города (sampled betweenness). Порядок для коммунальщиков = P(льда) × поток пешеходов.

In [16]:
city = graph.get_city()
pts = [(s['lon'], s['lat']) for s in city.stops] + [(p['lon'], p['lat']) for p in config.PLACES]
t0 = time.time()
store = reports.Store(city, pts, runs=900)
print(f'900 кратчайших путей за {time.time() - t0:.1f} с, демо-отметок: {len(store.items)}')
an = pipeline.analyse(city, weather.scenario('thaw_freeze'))
for i, q in enumerate(store.top(an['risks'], time.time()), 1):
    print(f"{i:2}. {(q['name'] or 'без названия'):38} ≈{q['len']:5} м  лёд {q['p']:.0%}  поток: {q['flow']:4}  отметок: {q['n']}")

900 кратчайших путей за 1.6 с, демо-отметок: 32
 1. улица Окжетпес                         ≈  893 м  лёд 86%  поток: high  отметок: 4
 2. улица Керей и Жанибек хандар           ≈ 2037 м  лёд 88%  поток: high  отметок: 14
 3. улица Бухар Жырау                      ≈ 1030 м  лёд 88%  поток: high  отметок: 1
 4. проспект Улы Дала                      ≈ 1144 м  лёд 80%  поток: high  отметок: 1
 5. улица Акан Серы                        ≈  695 м  лёд 89%  поток: high  отметок: 0
 6. улица Конституции                      ≈  558 м  лёд 81%  поток: high  отметок: 0
 7. проспект Мангилик Ел                   ≈  810 м  лёд 73%  поток: high  отметок: 0
 8. улица Акмешит                          ≈  657 м  лёд 92%  поток: high  отметок: 1
 9. улица Шабал Бейсековой                 ≈  416 м  лёд 60%  поток: high  отметок: 1
10. улица Туркестан                        ≈  594 м  лёд 82%  поток: high  отметок: 1


## 6. Таймер обморожения: A\* с ограничением по времени на холоде

Ветро-холод (формула Environment Canada): $W = 13.12 + 0.6215T - 11.37V^{0.16} + 0.3965TV^{0.16}$. Ниже −28° открытая кожа обмораживается за 10 мин, ниже −40° — за 5, ниже −48° — за 2.

In [17]:
from boran import frost
for t, v in ((-9, 4), (-16, 18), (-25, 8), (-35, 2), (-40, 10)):
    w, lim, band = frost.limit(t, v)
    print(f'{t:+4}°, ветер {v:2} м/с → ветро-холод {w:+6.1f}°  предел:', f'{lim} мин' if lim else 'нет')

  -9°, ветер  4 м/с → ветро-холод  -15.4°  предел: нет
 -16°, ветер 18 м/с → ветро-холод  -31.4°  предел: 10 мин
 -25°, ветер  8 м/с → ветро-холод  -38.9°  предел: 10 мин
 -35°, ветер  2 м/с → ветро-холод  -43.3°  предел: 5 мин
 -40°, ветер 10 м/с → ветро-холод  -60.1°  предел: 2 мин


Поиск идёт по меткам (узел, метры на холоде с последнего тепла): нельзя пройти участок, если счётчик превысит предел; в тёплом месте (ТЦ, кафе, аптека, супермаркет) счётчик обнуляется за 3 минуты. Доминируемые метки отбрасываются. Сценарий «Буран»: Байтерек → Хан Шатыр.

In [18]:
bz = weather.scenario('blizzard')
an_b = pipeline.analyse(city, bz)
P = {p['id']: (p['lon'], p['lat']) for p in config.PLACES}
a, b = city.nearest_node(*P['baiterek']), city.nearest_node(*P['khanshatyr'])
plain = astar.search(city, a, b, an_b['risks'], an_b['wind_k'])
w, lim, _ = frost.limit(bz['now']['temp'], bz['now']['wind'])
warm = frost.search(city, a, b, an_b['risks'], an_b['wind_k'], lim * 60 * config.WALK_SPEED)
print(f'ветро-холод {w}°, предел {lim} мин')
print(f"обычный A*:  {astar.describe(city, plain, an_b['risks'], an_b['wind_k'])['length']} м, на улице подряд {frost.minutes(frost.stretches(city, plain)):.0f} мин")
print(f"с теплом:    {astar.describe(city, warm, an_b['risks'], an_b['wind_k'])['length']} м, на улице подряд максимум {frost.minutes(frost.stretches(city, warm)):.1f} мин")
for s in warm['warm']:
    print(f"  через {s['d']:6.0f} м — погреться: {s['name'] or s['kind']} ({s['kind']})")

ветро-холод -31.4°, предел 10 мин
обычный A*:  2610 м, на улице подряд 31 мин
с теплом:    2734 м, на улице подряд максимум 9.8 мин
  через    824 м — погреться: Leto Cafe (cafe)
  через   1614 м — погреться: KFC (cafe)
  через   1938 м — погреться: Festival Avenue (mall)


## 7. Кто идёт: профили пешехода

Профиль меняет стоимость рёбер A\* (вес льда, лестницы, грунтовки, скорость) и цену падения в MDP.

In [19]:
mild = weather.scenario('thaw_freeze')
an_m = pipeline.analyse(city, mild)
for prof in ('base', 'elder', 'stroller', 'wheel'):
    r = pipeline.route(city, an_m, mild, P['baiterek'], P['khanshatyr'], prof)
    s = r['safe']
    print(f"{prof:8} {s['length']:5} м  {s['minutes']:5.1f} мин  риск {s['fall'] * 100:4.1f}%  лестниц: {r['steps']}  совет MDP: {r['mdp']['best']}")

base      2578 м   32.2 мин  риск  4.3%  лестниц: 2  совет MDP: go
elder     2572 м   45.1 мин  риск  4.3%  лестниц: 2  совет MDP: go
stroller  2572 м   39.2 мин  риск  4.3%  лестниц: 2  совет MDP: go
wheel     2572 м   50.1 мин  риск  4.3%  лестниц: 2  совет MDP: go


## 8. Ночь ИИ: диспетчер пескоразбрасывателей

HMM по часам ночи «оттепель → мороз» показывает, когда появится лёд. 6 машин выезжают в 00:00. Сравниваем две стратегии на реальном дорожном графе Астаны:

- **как сейчас** — машина едет на ближайшую большую улицу;
- **с ИИ** — машина выбирает отрезок с максимумом «предотвращённые травмы / минуты работы».

In [20]:
from boran import night
times, temps, tables, beliefs = night.night_tables()
for h, tmp, b in zip(times, temps, beliefs):
    print(h, f'{tmp:+4}°', 'P(гололёд) =', f'{b[4]:.2f}', '#' * int(b[4] * 40))

20:00 +3.0° P(гололёд) = 0.02 
21:00 +4.0° P(гололёд) = 0.01 
22:00 +4.0° P(гололёд) = 0.01 
23:00 +3.0° P(гололёд) = 0.02 
00:00 +2.0° P(гололёд) = 0.04 #
01:00 +1.0° P(гололёд) = 0.44 #################
02:00 +0.0° P(гололёд) = 0.71 ############################
03:00 -2.0° P(гололёд) = 0.72 ############################
04:00 -4.0° P(гололёд) = 0.65 #########################
05:00 -6.0° P(гололёд) = 0.58 #######################
06:00 -8.0° P(гололёд) = 0.53 #####################
07:00 -9.0° P(гололёд) = 0.49 ###################
08:00 -10.0° P(гололёд) = 0.46 ##################


In [21]:
t0 = time.time()
N = night.Night(city, reports.foot_traffic(city, pts, 900))
res = {k: N.run(k) for k in ('now', 'ai')}
print(f'отрезков улиц: {len(N.strokes)}, расчёт {time.time() - t0:.0f} с')
print(f"без машин:   {res['ai']['base']:.0f} травм утром")
for k, name in (('now', 'как сейчас'), ('ai', 'с ИИ')):
    r = res[k]
    print(f"{name:11}  {r['final']:.0f} травм  посыпано {r['km']} км  отрезков {r['n']}")
print('ИИ сберегает ещё', round(res['now']['final'] - res['ai']['final']), 'человек')

отрезков улиц: 4894, расчёт 76 с
без машин:   300 травм утром
как сейчас   196 травм  посыпано 537.3 км  отрезков 879
с ИИ         154 травм  посыпано 414.5 км  отрезков 828
ИИ сберегает ещё 42 человек


In [22]:
print('первые решения ИИ:')
for f in res['ai']['firsts'][:6]:
    print(f"  {f['t']}  {f['name'] or 'без названия':35} {f['len']:4} м   минус {f['saved']} травм")

первые решения ИИ:
  00:04  улица Бухар Жырау                    708 м   минус 1.0 травм
  00:10  улица Акмешит                        701 м   минус 1.2 травм
  00:11  улица Конституции                    714 м   минус 1.1 травм
  00:03  проспект Шaкaрима Кудайбердыулы      753 м   минус 0.9 травм
  00:04  шоссе Алаш                           869 м   минус 0.6 травм
  00:03  без названия                         626 м   минус 0.4 травм


## 9. Утро Астаны: 5 000 человек идут на работу

Толпа идёт кратчайшими путями (1 400 маршрутов дом → остановка, ТЦ, супермаркет). Шанс упасть на участке пропорционален длине × P(лёд утром); на посыпанных улицах P = 0.10. Для обеих стратегий одни и те же случайные числа, поэтому разница — только от того, где посыпали ночью.

In [23]:
t0 = time.time()
N2 = night.Night(city)
M = N2.export()
mo = M['morning']
print(f"маршрутов: {len(mo['paths'])}, людей: {len(mo['agents'])}, расчёт {time.time() - t0:.0f} с")
for k, name in (('none', 'без машин'), ('now', 'как сейчас'), ('ai', 'с ИИ')):
    print(f"{name:11} упало {mo['falls'][k]:4} человек")

маршрутов: 1400, людей: 5000, расчёт 74 с
без машин   упало  288 человек
как сейчас  упало  206 человек
с ИИ        упало  175 человек


## 10. Зал участвует: живой диспетчер по отметкам с телефонов

Каждая новая отметка проходит три шага:

1. **Байес:** P(лёд) участка до отметки → после. Меньше 40 % — ждём ещё одну отметку в радиусе 150 м.
2. **Обратный Дейкстра** от отметки по дорожному графу с односторонними улицами: одна волна даёт время до всех 12 машин. Выбираем минимум «когда освободится + время в пути» и останавливаем волну, как только она дальше лучшей машины.
3. **Очередь:** занятая машина приезжает после своей текущей улицы; если машина уже едет рядом, отметка присоединяется к ней.

In [24]:
from boran import hall
fleet = hall.Fleet(city)
print('машин:', len(fleet.trucks), '| перекрёстков в дорожном графе:', len(fleet.drive.adj))
an_l = pipeline.analyse(city, weather.scenario('thaw_freeze'))
spots = [('Байтерек', P['baiterek']), ('Хан Шатыр', P['khanshatyr']), ('Кампус', P['campus']), ('Общежитие', P['dorm'])]
for i, (name, (lon, lat)) in enumerate(spots):
    e = city.nearest_edge(lon, lat)
    prior = an_l['risks'][e]
    post = reports.observe(prior, 'ice')
    ev = fleet.handle({'id': f'm{i}', 'k': 'ice', 'lon': lon, 'lat': lat, 'e': e, 'es': [e], 'name': name}, post, 1000.0 + i, prior)
    j = fleet.jobs[-1] if ev['act'] == 'go' else None
    print(f"{name:10} Байес {prior:4.0%} → {post:4.0%}  {ev['act']:4}", f"машина {j['tr']:2}, в пути {j['t1'] - j['t0']:4.1f} мин, {j['km']} км" if j else '')

машин: 12 | перекрёстков в дорожном графе: 16929
Байтерек   Байес  72% →  96%  go   машина  1, в пути  9.2 мин, 4.61 км
Хан Шатыр  Байес  48% →  89%  go   машина  2, в пути 11.8 мин, 5.89 км
Кампус     Байес  48% →  89%  go   машина  3, в пути  7.3 мин, 3.65 км
Общежитие  Байес  72% →  96%  go   машина 11, в пути  3.8 мин, 1.88 км


In [25]:
lon, lat = P['baiterek']
f2 = hall.Fleet(city)
prior = 0.05
e1, e2 = city.nearest_edge(lon, lat), city.nearest_edge(lon + 0.0012, lat)
p1 = reports.observe(prior, 'ice')
a = f2.handle({'id': 'a', 'k': 'ice', 'lon': lon, 'lat': lat, 'e': e1, 'es': [e1], 'name': 'A'}, p1, 0.0, prior)
b = f2.handle({'id': 'b', 'k': 'ice', 'lon': lon + 0.0012, 'lat': lat, 'e': e2, 'es': [e2], 'name': 'B'}, p1, 5.0, prior)
print(f'сухой день, prior {prior:.0%}')
print('первая отметка «скользко»: ', a['act'], f"P = {a['p']:.0%}")
print('соседняя через 80 м:       ', b['act'], f"P = {b['p0']:.0%} → {b['p']:.0%} с учётом соседа → машина {b['tr']}")
print('первая отметка теперь:     ', f2.status['a']['act'], 'машина', f2.status['a']['tr'])

сухой день, prior 5%
первая отметка «скользко»:  wait P = 32%
соседняя через 80 м:        go P = 32% → 80% с учётом соседа → машина 1
первая отметка теперь:      go машина 1


In [26]:
import random
f3 = hall.Fleet(city, per_depot=1)
rnd = random.Random(3)
nodes = sorted(f3.drive.adj)
for i in range(14):
    lon, lat = city.nodes[rnd.choice(nodes)]
    e = city.nearest_edge(lon, lat)
    f3.handle({'id': f'q{i}', 'k': 'ice', 'lon': lon, 'lat': lat, 'e': e, 'es': [e], 'name': ''}, 0.9, i * 0.5)
print('6 машин, 14 отметок за 7 секунд:')
for j in f3.jobs:
    print(f"  работа {j['id']:2}  машина {j['tr']}  {'в очереди' if j['q'] else 'сразу    '}  выезд {j['t0']:5.1f}  на месте {j['t1']:5.1f}  готово {j['t2']:5.1f}")

6 машин, 14 отметок за 7 секунд:
  работа  1  машина 3  сразу      выезд   2.5  на месте   3.6  готово   6.6
  работа  2  машина 3  в очереди  выезд   6.6  на месте  20.0  готово  23.0
  работа  3  машина 6  сразу      выезд   3.5  на месте  10.0  готово  13.0
  работа  4  машина 1  сразу      выезд   4.0  на месте  14.5  готово  17.5
  работа  5  машина 4  сразу      выезд   4.5  на месте   6.2  готово   9.2
  работа  6  машина 4  в очереди  выезд   9.2  на месте  16.8  готово  19.8
  работа  7  машина 2  сразу      выезд   5.5  на месте  24.1  готово  27.1
  работа  8  машина 1  в очереди  выезд  17.5  на месте  31.8  готово  34.8
  работа  9  машина 5  сразу      выезд   6.5  на месте  20.4  готово  23.4
  работа 10  машина 6  в очереди  выезд  13.0  на месте  42.1  готово  45.1
  работа 11  машина 4  в очереди  выезд  19.8  на месте  39.9  готово  43.8
  работа 12  машина 5  в очереди  выезд  23.4  на месте  27.9  готово  30.9
  работа 13  машина 2  в очереди  выезд  27.1  на месте

## Итог

0. Пять алгоритмов поиска (BFS, DFS, UCS, Greedy, A\*) сравнены на реальном графе улиц Астаны.
1. HMM по погоде восстановил скрытое покрытие: после оттепели и ночного мороза вероятнее всего гололёд.
2. Байес превратил это в риск для каждой улицы: двор в тени опаснее чищеного проспекта.
3. A\* выбрал путь длиннее, но безопаснее.
4. MDP посоветовал переждать буран, потому что он стихает.
5. Отметки людей обновляют вероятность льда (Байес + HMM), а выборка кратчайших путей показывает городу, где посыпать первым.
6. A\* с ограничением по времени на холоде ведёт через тёплые места, чтобы ни один отрезок не превышал порог обморожения.
7. Профиль пешехода меняет маршрут и совет MDP.
8. ИИ-диспетчер теми же 6 машинами даёт меньше травм утром, чем «ближайшая большая улица».
9. Утром 5 000 человек идут по тем же улицам: после ночи с ИИ падает меньше людей, чем «как сейчас».
10. Отметки зрителей с телефонов проходят Байес, обратный Дейкстра и очередь — машина выезжает через секунды.